# 13 — Mine hard-positive candidates for the frozen v5 gate

Notebook 12 showed the new gate rejects unsupported motion well but also rejects many real supported exercises. This **CPU notebook** looks for supported examples that the frozen v5 gate finds difficult.

It uses the public Real-Time Exercise Recognition Dataset, which contains squat, push-up, bicep curl, and shoulder press from synthetic and real sources. The dataset card also says part of it came from Workout/Fitness, so this notebook does **not** trust every item as new evidence. It:

1. inventories video files or frame sequences without mixing source clips;
2. extracts the same normalized 17-joint MediaPipe representation;
3. scores each candidate with the frozen v5 gate;
4. flags exact or pose-near duplicates of the original Workout/Fitness landmark archive;
5. selects low-gate supported candidates plus a few easy controls;
6. exports landmarks, a manifest, contact sheets, and a review template.

This four-class source does not contain plank, so it cannot by itself repair the locked benchmark's plank failure; independent plank evidence remains a separate required batch.

The output is `calorie_hard_positive_candidates_v1.zip`. **Every row is `needs_visual_review` and `training_eligible=False`.** Notebook 14 must refuse it until the selected contact sheets and source labels are reviewed. The locked GainsByBrains video and labels are never downloaded.

Use a **CPU runtime**. A GPU does not materially speed up MediaPipe here. Expect about 3.1 GB of download traffic and a long first run because the source contains many files.

In [ ]:
%pip install -q --upgrade kagglehub mediapipe==0.10.35 opencv-python-headless pandas==2.2.3 tqdm

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import os
import re
import shutil
import urllib.request
import zipfile

import cv2
import kagglehub
import mediapipe as mp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from google.colab import files
from tqdm.auto import tqdm

SEED = 42
TARGET_FPS = 15
ASSUMED_IMAGE_FPS = 30
WINDOW = 64
STRIDE = 32
MAX_SOURCE_GROUPS_PER_CLASS = 40
MAX_SAMPLED_FRAMES_PER_GROUP = 900
HARD_CANDIDATES_PER_CLASS = 12
CONTROL_CANDIDATES_PER_CLASS = 3
MIN_DETECTION_RATE = 0.80
DUPLICATE_MSE_THRESHOLD = 1e-5

OUTPUT_ROOT = Path('/content/calorie_hard_positive_candidates_v1')
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True)

np.random.seed(SEED)
print('Runtime: CPU is recommended')
print('MediaPipe:', mp.__version__)
os.system('df -h /content | tail -1')

## Download and inventory the independent candidate pool

`kagglehub` keeps the data in the temporary Colab runtime. The code supports either original videos or directories of extracted image frames. A frame directory is treated as one source group; it is never joined with another directory.

If the upstream folder layout changes and source grouping becomes ambiguous, the notebook stops instead of silently mixing people or videos.

In [ ]:
DATASET_ROOT = Path(kagglehub.dataset_download(
    'riccardoriccio/real-time-exercise-recognition-dataset'
))
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.webm', '.mkv'}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
CLASS_ALIASES = {
    'squat': ('squat',),
    'push_up': ('pushup', 'pushups'),
    'bicep_curl': ('barbellbicepscurl', 'bicepscurl', 'bicepcurl'),
    'shoulder_press': ('shoulderpress',),
}


def compact_text(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).lower())


def infer_class(path):
    pieces = [compact_text(piece) for piece in path.parts]
    for canonical, aliases in CLASS_ALIASES.items():
        if any(alias in piece for piece in pieces for alias in aliases):
            return canonical
    return None


def frame_prefix(path):
    stem = path.stem
    prefix = re.sub(
        r'(?i)(?:[_-]?(?:frame|img|image))?[_-]?\d{1,8}$', '', stem
    ).strip('_- ')
    return prefix or '__single_sequence__'


def spread_sample(items, limit):
    items = list(items)
    if limit is None or len(items) <= limit:
        return items
    indices = np.linspace(0, len(items) - 1, num=limit, dtype=int)
    return [items[index] for index in sorted(set(indices.tolist()))]


all_files = [path for path in DATASET_ROOT.rglob('*') if path.is_file()]
video_files = [path for path in all_files if path.suffix.lower() in VIDEO_EXTENSIONS]
image_files = [path for path in all_files if path.suffix.lower() in IMAGE_EXTENSIONS]
print('Dataset root:', DATASET_ROOT)
print('All files:', len(all_files), 'videos:', len(video_files), 'images:', len(image_files))

candidates = []
video_counts = pd.Series([infer_class(path) for path in video_files]).value_counts()

# Prefer original video containers when the dataset supplies enough of them.
use_videos = all(video_counts.get(name, 0) >= 3 for name in CLASS_ALIASES)
if use_videos:
    for path in video_files:
        canonical = infer_class(path)
        if canonical:
            candidates.append({
                'kind': 'video', 'canonical_class': canonical,
                'source_ref': str(path.relative_to(DATASET_ROOT)),
                'video_path': path, 'frame_paths': None,
            })
else:
    grouped = defaultdict(list)
    for path in image_files:
        canonical = infer_class(path)
        if canonical is None:
            continue
        # The parent directory is the primary source-video boundary. A stable
        # filename prefix separates clips only when several clips share a folder.
        key = (canonical, str(path.parent.relative_to(DATASET_ROOT)), frame_prefix(path))
        grouped[key].append(path)

    parent_prefix_counts = defaultdict(set)
    for canonical, parent, prefix in grouped:
        parent_prefix_counts[(canonical, parent)].add(prefix)

    for (canonical, parent, prefix), paths in grouped.items():
        prefixes = parent_prefix_counts[(canonical, parent)]
        if len(prefixes) == 1:
            group_ref = parent
        else:
            group_ref = f'{parent}::{prefix}'
        paths = sorted(paths, key=lambda p: [
            int(part) if part.isdigit() else part.lower()
            for part in re.split(r'(\d+)', p.name)
        ])
        if len(paths) >= 20:
            candidates.append({
                'kind': 'frames', 'canonical_class': canonical,
                'source_ref': group_ref,
                'video_path': None, 'frame_paths': paths,
            })

inventory = pd.DataFrame([{
    'kind': item['kind'],
    'canonical_class': item['canonical_class'],
    'source_ref': item['source_ref'],
    'source_frames': len(item['frame_paths']) if item['frame_paths'] else np.nan,
} for item in candidates])
print('\nDiscovered source groups:')
display(inventory.groupby(['kind', 'canonical_class']).size().rename('groups').reset_index())

assert len(candidates), 'No supported videos or frame sequences were discovered.'
for canonical in CLASS_ALIASES:
    count = int((inventory.canonical_class == canonical).sum())
    assert count >= 3, (
        f'Only {count} source groups found for {canonical}. '
        'The upstream layout is ambiguous; inspect inventory before continuing.'
    )

sampled_candidates = []
for canonical in CLASS_ALIASES:
    group = [item for item in candidates if item['canonical_class'] == canonical]
    group = sorted(group, key=lambda item: item['source_ref'])
    sampled_candidates.extend(spread_sample(group, MAX_SOURCE_GROUPS_PER_CLASS))
print('Groups selected for mining:', len(sampled_candidates))

## Load the frozen v5 gate and original training signatures

The checkpoint and original Workout/Fitness landmarks come from the versioned GitHub artifacts. The checkpoint is used only to rank candidates. Original landmarks provide a conservative duplicate check. Low distance is a warning, not proof of independence.

In [ ]:
BRANCH_ROOT = (
    'https://raw.githubusercontent.com/bariscelikk1/cALorie/'
    'feat/tcn-rebuild/ml/artifacts'
)
MODEL_PATH = Path('/content/two_stage_tcn_v5.pt')
V1_ZIP = Path('/content/calorie_workoutfitness_landmarks_v1.zip')


def download(url, destination):
    request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request, timeout=300) as response, destination.open('wb') as output:
        shutil.copyfileobj(response, output)
    print(destination.name, round(destination.stat().st_size / 1e6, 2), 'MB')


download(f'{BRANCH_ROOT}/v5/model/two_stage_tcn.pt', MODEL_PATH)
download(f'{BRANCH_ROOT}/v1/landmarks/calorie_workoutfitness_landmarks_v1.zip', V1_ZIP)


class TemporalBlock(torch.nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = torch.nn.Sequential(
            torch.nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            torch.nn.BatchNorm1d(out_channels), torch.nn.ReLU(), torch.nn.Dropout(dropout),
            torch.nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            torch.nn.BatchNorm1d(out_channels), torch.nn.ReLU(), torch.nn.Dropout(dropout),
        )
        self.residual = (
            torch.nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else torch.nn.Identity()
        )

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(torch.nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = torch.nn.Sequential(*blocks)
        self.classifier = torch.nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
GATE_THRESHOLD = float(checkpoint['gate_threshold'])
WINDOW = int(checkpoint['window_frames'])
gate_model = SmallTCN(int(checkpoint['input_features']), 2)
gate_model.load_state_dict(checkpoint['gate_state_dict'])
gate_model.eval()
print('Frozen checkpoint:', checkpoint['experiment'])
print('Gate threshold:', GATE_THRESHOLD, 'window:', WINDOW)

V1_ROOT = Path('/content/workoutfitness_v1_reference')
if V1_ROOT.exists():
    shutil.rmtree(V1_ROOT)
V1_ROOT.mkdir()
with zipfile.ZipFile(V1_ZIP) as archive:
    archive.extractall(V1_ROOT)

v1_manifest_path = next(V1_ROOT.rglob('manifest.csv'))
v1_manifest = pd.read_csv(v1_manifest_path).fillna('')
V1_FINGERPRINTS = set(v1_manifest.get('fingerprint', pd.Series(dtype=str)).astype(str)) - {''}


def fixed_signature(sequence, length=64):
    sequence = np.asarray(sequence, dtype=np.float32)
    old_x = np.linspace(0, 1, len(sequence))
    new_x = np.linspace(0, 1, length)
    flat = sequence[:, :, :2].reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, index])
        for index in range(flat.shape[1])
    ], axis=1)
    return sampled.astype(np.float32)


reference_signatures = defaultdict(list)
for row in v1_manifest.itertuples(index=False):
    canonical = str(row.canonical_class)
    npz_path = v1_manifest_path.parent / str(row.output_path)
    if canonical in CLASS_ALIASES and npz_path.exists():
        with np.load(npz_path, allow_pickle=False) as item:
            reference_signatures[canonical].append(fixed_signature(item['landmarks']))
print('Reference signatures:', {key: len(value) for key, value in reference_signatures.items()})

## Extract normalized pose and score source groups

MediaPipe samples at 15 FPS. The same 17 joints, hip-centering, torso scaling, 64-frame windows, and x/y input used during v5 training are preserved. The gate outputs `supported_probability`; lower values mean a stronger hard-positive candidate.

In [ ]:
MODEL_ASSET = Path('/content/pose_landmarker_full.task')
MODEL_URL = (
    'https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
    'pose_landmarker_full/float16/latest/pose_landmarker_full.task'
)
if not MODEL_ASSET.exists():
    urllib.request.urlretrieve(MODEL_URL, MODEL_ASSET)

KEEP_17 = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_HIP, RIGHT_HIP = 11, 12

options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_ASSET)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False,
)


def normalize_pose(frame):
    xyz = frame[:, :3].copy()
    hip_center = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2.0
    shoulder_center = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2.0
    shoulder_width = np.linalg.norm(xyz[LEFT_SHOULDER] - xyz[RIGHT_SHOULDER])
    torso_length = np.linalg.norm(shoulder_center - hip_center)
    scale = max(float(shoulder_width), float(torso_length), 1e-6)
    xyz = (xyz - hip_center) / scale
    return np.concatenate([xyz, frame[:, 3:4]], axis=1).astype(np.float32)


def fill_missing(sequence):
    shape = sequence.shape
    flat = sequence.reshape(shape[0], -1)
    filled = pd.DataFrame(flat).interpolate(limit_direction='both').fillna(0).to_numpy()
    return filled.reshape(shape).astype(np.float32)


def frame_pose(frame_bgr, landmarker, timestamp_ms):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect_for_video(image, timestamp_ms)
    if not result.pose_landmarks:
        return None
    points = result.pose_landmarks[0]
    pose = np.asarray([
        [points[i].x, points[i].y, points[i].z, float(points[i].visibility or 0.0)]
        for i in KEEP_17
    ], dtype=np.float32)
    return normalize_pose(pose)


def file_fingerprint(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    size = path.stat().st_size
    digest.update(str(size).encode())
    with path.open('rb') as handle:
        digest.update(handle.read(block_size))
        if size > block_size:
            handle.seek(max(0, size - block_size))
            digest.update(handle.read(block_size))
    return digest.hexdigest()


def extract_candidate(candidate):
    poses, detected = [], 0
    fingerprint = ''
    with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
        if candidate['kind'] == 'video':
            path = candidate['video_path']
            fingerprint = file_fingerprint(path)
            capture = cv2.VideoCapture(str(path))
            if not capture.isOpened():
                return None, 0.0, fingerprint
            source_fps = float(capture.get(cv2.CAP_PROP_FPS) or 0) or 30.0
            interval = max(source_fps / TARGET_FPS, 1.0)
            source_index, next_index = 0, 0.0
            while len(poses) < MAX_SAMPLED_FRAMES_PER_GROUP:
                ok, frame = capture.read()
                if not ok:
                    break
                if source_index + 1e-9 >= next_index:
                    timestamp_ms = int(round(len(poses) * 1000 / TARGET_FPS))
                    pose = frame_pose(frame, landmarker, timestamp_ms)
                    poses.append(pose if pose is not None else np.full((17, 4), np.nan, np.float32))
                    detected += int(pose is not None)
                    next_index += interval
                source_index += 1
            capture.release()
        else:
            paths = candidate['frame_paths'][::max(1, round(ASSUMED_IMAGE_FPS / TARGET_FPS))]
            paths = spread_sample(paths, MAX_SAMPLED_FRAMES_PER_GROUP)
            for frame_index, path in enumerate(paths):
                frame = cv2.imread(str(path))
                timestamp_ms = int(round(frame_index * 1000 / TARGET_FPS))
                pose = frame_pose(frame, landmarker, timestamp_ms) if frame is not None else None
                poses.append(pose if pose is not None else np.full((17, 4), np.nan, np.float32))
                detected += int(pose is not None)

    if not poses:
        return None, 0.0, fingerprint
    return fill_missing(np.stack(poses)), detected / len(poses), fingerprint


def gate_scores(sequence):
    starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and starts[-1] != len(sequence) - WINDOW:
        starts.append(len(sequence) - WINDOW)
    windows = []
    for start in starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([window, np.repeat(window[-1:], WINDOW - len(window), axis=0)])
        windows.append(window[:, :, :2].reshape(WINDOW, -1).T)
    features = torch.from_numpy(np.asarray(windows, dtype=np.float32))
    with torch.no_grad():
        probability = gate_model(features).softmax(1)[:, 1].numpy()
    return probability


def nearest_reference_mse(canonical, sequence):
    refs = reference_signatures.get(canonical, [])
    if not refs:
        return np.nan
    signature = fixed_signature(sequence)
    return float(min(np.mean((signature - ref) ** 2) for ref in refs))


results = []
sequences = {}
for candidate in tqdm(sampled_candidates, desc='Mining source groups'):
    sequence, detection_rate, fingerprint = extract_candidate(candidate)
    if sequence is None:
        continue
    probability = gate_scores(sequence)
    candidate_id = hashlib.sha1(candidate['source_ref'].encode()).hexdigest()[:16]
    nearest_mse = nearest_reference_mse(candidate['canonical_class'], sequence)
    source_hint = compact_text(candidate['source_ref'])
    duplicate_hint = any(token in source_hint for token in (
        'workoutfitness', 'hasyimabdillah', 'kaggleworkout'
    ))
    exact_duplicate = bool(fingerprint and fingerprint in V1_FINGERPRINTS)
    pose_near_duplicate = bool(np.isfinite(nearest_mse) and nearest_mse < DUPLICATE_MSE_THRESHOLD)
    suspected_duplicate = duplicate_hint or exact_duplicate or pose_near_duplicate
    sequences[candidate_id] = sequence
    results.append({
        'clip_id': candidate_id,
        'source_dataset': 'real-time-exercise-recognition-dataset',
        'source_ref': candidate['source_ref'],
        'source_kind': candidate['kind'],
        'canonical_class': candidate['canonical_class'],
        'group_id': f'realtime-exercise:{candidate_id}',
        'sampled_frames': len(sequence),
        'detection_rate': detection_rate,
        'window_count': len(probability),
        'gate_supported_mean': float(probability.mean()),
        'gate_supported_min': float(probability.min()),
        'gate_supported_p10': float(np.quantile(probability, 0.10)),
        'gate_rejected_fraction': float((probability < GATE_THRESHOLD).mean()),
        'nearest_v1_pose_mse': nearest_mse,
        'exact_v1_fingerprint': exact_duplicate,
        'duplicate_source_hint': duplicate_hint,
        'suspected_duplicate': suspected_duplicate,
        'source_fingerprint': fingerprint,
    })

score_frame = pd.DataFrame(results)
assert len(score_frame), 'No candidate produced landmarks.'
print('Scored groups:', len(score_frame))
display(score_frame.groupby('canonical_class').agg(
    groups=('clip_id', 'count'),
    detection_mean=('detection_rate', 'mean'),
    supported_mean=('gate_supported_mean', 'mean'),
    rejected_mean=('gate_rejected_fraction', 'mean'),
    suspected_duplicates=('suspected_duplicate', 'sum'),
).round(3))

## Select difficult positives and build review material

A valid hard positive has a known supported class, reliable pose detection, and a high fraction of windows rejected by the gate. The notebook keeps the lowest-scoring candidates per class and a few easy controls. Controls tell us whether review quality differs between hard and ordinary examples.

Selection does not approve a label. Contact sheets must be checked before any candidate enters training.

In [ ]:
eligible = score_frame[
    (score_frame.detection_rate >= MIN_DETECTION_RATE)
    & (~score_frame.suspected_duplicate)
].copy()
assert len(eligible), 'Every candidate was low-detection or a suspected duplicate.'

selected_ids = []
roles = {}
for canonical, group in eligible.groupby('canonical_class'):
    hardest = group.sort_values(
        ['gate_supported_mean', 'gate_rejected_fraction'],
        ascending=[True, False],
    ).head(HARD_CANDIDATES_PER_CLASS)
    remaining = group[~group.clip_id.isin(hardest.clip_id)]
    controls = remaining.sort_values('gate_supported_mean', ascending=False).head(
        CONTROL_CANDIDATES_PER_CLASS
    )
    for clip_id in hardest.clip_id:
        row = hardest[hardest.clip_id == clip_id].iloc[0]
        roles[clip_id] = (
            'hard_positive' if row.gate_rejected_fraction >= 0.5
            else 'lowest_available_positive'
        )
    for clip_id in controls.clip_id:
        roles[clip_id] = 'positive_control'
    selected_ids.extend(hardest.clip_id.tolist() + controls.clip_id.tolist())

selected = score_frame[score_frame.clip_id.isin(selected_ids)].copy()
selected['mining_role'] = selected.clip_id.map(roles)
selected['review_status'] = 'needs_visual_review'
selected['training_eligible'] = False
selected['license'] = 'CC-BY-NC-SA-4.0 (dataset card)'
selected['model_checkpoint'] = 'v5_hard_negative_two_stage_tcn'
selected['gate_threshold'] = GATE_THRESHOLD

candidate_lookup = {
    hashlib.sha1(item['source_ref'].encode()).hexdigest()[:16]: item
    for item in sampled_candidates
}


def contact_frames(candidate, count=6):
    frames = []
    if candidate['kind'] == 'video':
        capture = cv2.VideoCapture(str(candidate['video_path']))
        total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        for index in np.linspace(0, max(total - 1, 0), count, dtype=int):
            capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))
            ok, frame = capture.read()
            if ok:
                frames.append(frame)
        capture.release()
    else:
        paths = candidate['frame_paths']
        for index in np.linspace(0, len(paths) - 1, count, dtype=int):
            frame = cv2.imread(str(paths[index]))
            if frame is not None:
                frames.append(frame)
    return frames


def save_contact_sheet(frames, title, path):
    if not frames:
        return False
    tile_w, tile_h = 320, 240
    tiles = []
    for frame in frames:
        resized = cv2.resize(frame, (tile_w, tile_h))
        tiles.append(resized)
    while len(tiles) < 6:
        tiles.append(np.zeros_like(tiles[0]))
    sheet = np.vstack([np.hstack(tiles[:3]), np.hstack(tiles[3:6])])
    cv2.rectangle(sheet, (0, 0), (sheet.shape[1], 34), (0, 0, 0), -1)
    cv2.putText(sheet, title[:110], (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    path.parent.mkdir(parents=True, exist_ok=True)
    return cv2.imwrite(str(path), sheet)

manifest_rows = []
for row in selected.itertuples(index=False):
    sequence = sequences[row.clip_id]
    output_path = OUTPUT_ROOT / row.canonical_class / f'{row.clip_id}.npz'
    output_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        output_path,
        landmarks=sequence.astype(np.float32),
        label=row.canonical_class,
        source_dataset=row.source_dataset,
        source_ref=row.source_ref,
        target_fps=TARGET_FPS,
    )
    sheet_path = OUTPUT_ROOT / 'contact_sheets' / f'{row.clip_id}.jpg'
    save_contact_sheet(
        contact_frames(candidate_lookup[row.clip_id]),
        f'{row.canonical_class} | {row.mining_role} | gate={row.gate_supported_mean:.3f}',
        sheet_path,
    )
    values = row._asdict()
    values['output_path'] = str(output_path.relative_to(OUTPUT_ROOT))
    values['contact_sheet'] = str(sheet_path.relative_to(OUTPUT_ROOT))
    values['status'] = 'ok'
    values['source_fps'] = TARGET_FPS
    manifest_rows.append(values)

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_ROOT / 'manifest.csv', index=False)
score_frame.drop(columns=[], errors='ignore').to_csv(OUTPUT_ROOT / 'all_candidate_scores.csv', index=False)
inventory.to_csv(OUTPUT_ROOT / 'source_inventory.csv', index=False)

review = manifest[[
    'clip_id', 'canonical_class', 'mining_role', 'contact_sheet',
    'gate_supported_mean', 'gate_rejected_fraction', 'suspected_duplicate',
]].copy()
review['approve_for_training'] = False
review['corrected_class'] = ''
review['reviewer'] = ''
review['review_notes'] = ''
review.to_csv(OUTPUT_ROOT / 'review_template.csv', index=False)

print('Selected candidates:', len(manifest))
display(manifest.groupby(['canonical_class', 'mining_role']).size().rename('clips').reset_index())
assert manifest.training_eligible.eq(False).all()
assert manifest.review_status.eq('needs_visual_review').all()
assert manifest.clip_id.is_unique
assert manifest.group_id.is_unique
assert all((OUTPUT_ROOT / path).exists() for path in manifest.output_path)
assert all((OUTPUT_ROOT / path).exists() for path in manifest.contact_sheet)

## Audit and download

The ZIP deliberately cannot be fed straight into training. Fill `review_template.csv` after inspecting every contact sheet. A later promotion step will require `approve_for_training=True`, a reviewer name, no unresolved duplicate warning, and the corrected canonical class when needed.

In [ ]:
summary = {
    'source_dataset': 'riccardoriccio/real-time-exercise-recognition-dataset',
    'source_license': 'CC-BY-NC-SA-4.0 (dataset card)',
    'checkpoint': checkpoint['experiment'],
    'gate_threshold': GATE_THRESHOLD,
    'scored_source_groups': int(len(score_frame)),
    'selected_candidates': int(len(manifest)),
    'selected_by_class': manifest.canonical_class.value_counts().sort_index().to_dict(),
    'hard_positive_candidates': int((manifest.mining_role == 'hard_positive').sum()),
    'suspected_duplicates_excluded': int(score_frame.suspected_duplicate.sum()),
    'training_ready': False,
    'required_next_step': 'human visual review, duplicate adjudication, and explicit promotion',
    'locked_gainsbybrains_used': False,
}
(OUTPUT_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2))
(OUTPUT_ROOT / 'README.txt').write_text(
    'Hard-positive candidate archive only.\n'
    'Inspect contact_sheets/, complete review_template.csv, and keep unapproved rows out of training.\n'
    'The frozen GainsByBrains benchmark was not used.\n'
)
print(json.dumps(summary, indent=2))

# Preview a small balanced sample inside Colab.
preview = manifest.groupby('canonical_class', group_keys=False).head(2)
fig, axes = plt.subplots(len(preview), 1, figsize=(14, max(4, 3 * len(preview))))
axes = np.atleast_1d(axes)
for axis, row in zip(axes, preview.itertuples(index=False)):
    image = cv2.cvtColor(cv2.imread(str(OUTPUT_ROOT / row.contact_sheet)), cv2.COLOR_BGR2RGB)
    axis.imshow(image)
    axis.set_title(
        f'{row.canonical_class} | {row.mining_role} | '
        f'mean gate={row.gate_supported_mean:.3f} | rejected={row.gate_rejected_fraction:.1%}'
    )
    axis.axis('off')
plt.tight_layout()
plt.show()

zip_path = shutil.make_archive(
    '/content/calorie_hard_positive_candidates_v1', 'zip', root_dir=OUTPUT_ROOT
)
print('ZIP ready:', zip_path)
print('ZIP size MB:', round(Path(zip_path).stat().st_size / 1024**2, 2))
files.download(zip_path)

## What comes next

Send back `calorie_hard_positive_candidates_v1.zip`. We will inspect the manifest, score distribution, duplicate flags, and completed review decisions. Only approved independent rows will be promoted into a new versioned landmark archive. Notebook 14 will then perform a controlled retrain with the same architecture and a calibration split that remains separate from the locked GainsByBrains test.